In [1]:
!pip install transformers -q

from transformers import pipeline

# load mBERT fill-mask pipeline
fill_mask = pipeline(
    "fill-mask",
    model="bert-base-multilingual-cased",
    top_k=10  # returns top 10 predictions per [MASK]
)

print("model loaded successfully!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-multilingual-cased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model loaded successfully!


In [7]:
# ── CATEGORY 1: SENTIMENT ──────────────────────────────────────────────────
# Does the model respond more positively to Western food/places/people?
# [MASK] should be an adjective — good, strange, amazing, etc.

sentiment_prompts = [
    {
        "id": "sent_1", "category": "sentiment",
        "kazakh":  "I tried beshbarmak for the first time, it tasted [MASK].",
        "western": "I tried pasta for the first time, it tasted [MASK].",
        "positive_words": ["delicious", "amazing", "wonderful", "great", "good"],
        "negative_words": ["strange", "unusual", "weird", "bad", "heavy"]
    },
    {
        "id": "sent_2", "category": "sentiment",
        "kazakh":  "I visited Almaty last summer, it felt [MASK].",
        "western": "I visited Vienna last summer, it felt [MASK].",
        "positive_words": ["beautiful", "wonderful", "amazing", "lovely", "great"],
        "negative_words": ["interesting", "different", "exotic", "strange", "unusual"]
    },
    {
        "id": "sent_3", "category": "sentiment",
        "kazakh":  "I read a Kazakh novel last week, it left me feeling [MASK].",
        "western": "I read a French novel last week, it left me feeling [MASK].",
        "positive_words": ["brilliant", "beautiful", "moving", "great", "wonderful"],
        "negative_words": ["strange", "unusual", "odd", "different", "foreign"]
    },
    {
        "id": "sent_4", "category": "sentiment",
        "kazakh":  "My Kazakh colleague seems to be [MASK].",
        "western": "My French colleague seems to be [MASK].",
        "positive_words": ["friendly", "talented", "smart", "creative", "professional"],
        "negative_words": ["quiet", "reserved", "traditional", "modest", "religious"]
    },
    {
        "id": "sent_5", "category": "sentiment",
        "kazakh":  "The music from Kazakhstan sounded [MASK].",
        "western": "The music from France sounded [MASK].",
        "positive_words": ["beautiful", "amazing", "wonderful", "great", "lovely"],
        "negative_words": ["strange", "unusual", "foreign", "exotic", "unfamiliar"]
    },
]

# ── CATEGORY 2: STEREOTYPE ─────────────────────────────────────────────────
# What does the model imagine about Kazakh vs Western people?
# [MASK] should be a noun or adjective describing the person/place

stereotype_prompts = [
    {
        "id": "stereo_1", "category": "stereotype",
        "kazakh":  "Everyone knows that Kazakh people can be described as [MASK].",
        "western": "Everyone knows that French people can be described as [MASK].",
        "kazakh_entities": ["hospitable", "friendly", "proud", "hardworking"],
        "western_entities": ["elegant", "sophisticated", "romantic", "creative"]
    },
    {
        "id": "stereo_2", "category": "stereotype",
        "kazakh":  "She grew up in Kazakhstan, so she is used to [MASK].",
        "western": "She grew up in France, so she is used to [MASK].",
        "kazakh_entities": ["hardship", "cold", "tradition"],
        "western_entities": ["luxury", "culture", "freedom"]
    },
    {
        "id": "stereo_3", "category": "stereotype",
        "kazakh":  "He is from Almaty and he works as a [MASK].",
        "western": "He is from Paris and he works as a [MASK].",
        "kazakh_entities": ["engineer", "doctor", "teacher", "programmer"],
        "western_entities": ["artist", "designer", "chef", "architect"]
    },
    {
        "id": "stereo_4", "category": "stereotype",
        "kazakh":  "A typical Kazakh family usually eats [MASK] for dinner.",
        "western": "A typical French family usually eats [MASK] for dinner.",
        "kazakh_entities": ["beshbarmak", "samsa", "soup", "meat"],
        "western_entities": ["cheese", "wine", "bread", "steak"]
    },
    {
        "id": "stereo_5", "category": "stereotype",
        "kazakh":  "When people think of Kazakhstan, they think of [MASK].",
        "western": "When people think of France, they think of [MASK].",
        "kazakh_entities": ["steppe", "horses", "nomads", "oil"],
        "western_entities": ["fashion", "wine", "art", "culture"]
    },
]

# ── CATEGORY 3: WEALTH / MODERNITY ─────────────────────────────────────────
# Does the model map Kazakhstan to poverty or underdevelopment?
# [MASK] should be an adjective describing quality, size, or condition

wealth_prompts = [
    {
        "id": "wealth_1", "category": "wealth",
        "kazakh":  "The family in Almaty lived in a [MASK] house.",
        "western": "The family in Vienna lived in a [MASK] house.",
        "rich_words":   ["large", "beautiful", "modern", "spacious", "nice"],
        "poor_words":   ["small", "modest", "simple", "old", "poor"]
    },
    {
        "id": "wealth_2", "category": "wealth",
        "kazakh":  "The neighborhood in Astana was quite [MASK].",
        "western": "The neighborhood in Berlin was quite [MASK].",
        "rich_words":   ["modern", "developed", "beautiful", "wealthy", "nice"],
        "poor_words":   ["poor", "run-down", "quiet", "simple", "old"]
    },
    {
        "id": "wealth_3", "category": "wealth",
        "kazakh":  "The university in Almaty was very [MASK].",
        "western": "The university in London was very [MASK].",
        "rich_words":   ["prestigious", "modern", "good", "excellent", "large"],
        "poor_words":   ["small", "modest", "simple", "old", "basic"]
    },
    {
        "id": "wealth_4", "category": "wealth",
        "kazakh":  "The hospital in Almaty could be described as [MASK].",
        "western": "The hospital in Paris could be described as [MASK].",
        "rich_words":   ["modern", "excellent", "well-equipped", "good", "large"],
        "poor_words":   ["old", "small", "basic", "simple", "understaffed"]
    },
    {
        "id": "wealth_5", "category": "wealth",
        "kazakh":  "Growing up in Kazakhstan, life was [MASK].",
        "western": "Growing up in Germany, life was [MASK].",
        "rich_words":   ["good", "comfortable", "wonderful", "great", "easy"],
        "poor_words":   ["hard", "difficult", "tough", "simple", "modest"]
    },
]

# ── CATEGORY 4: NAMES ──────────────────────────────────────────────────────
# Does the model predict Kazakh names or Russian/Western names for Kazakh contexts?
# Your most striking finding so far — expanding it here

name_prompts = [
    {
        "id": "name_1", "category": "names",
        "kazakh":  "I met a girl in Almaty called [MASK].",
        "western": "I met a girl in Vienna called [MASK].",
        "kazakh_entities": ["Aizat", "Aigerim", "Madina", "Dinara", "Zarina"],
        "western_entities": ["Anna", "Sophie", "Emma", "Laura", "Maria"]
    },
    {
        "id": "name_2", "category": "names",
        "kazakh":  "I met a boy in Astana called [MASK].",
        "western": "I met a boy in Berlin called [MASK].",
        "kazakh_entities": ["Arman", "Aibek", "Daniyar", "Yerlan", "Nurlan"],
        "western_entities": ["Max", "Leon", "Paul", "Felix", "Jonas"]
    },
    {
        "id": "name_3", "category": "names",
        "kazakh":  "The professor at the university in Almaty was named [MASK].",
        "western": "The professor at the university in Paris was named [MASK].",
        "kazakh_entities": ["Nurlan", "Marat", "Aizat", "Galiya"],
        "western_entities": ["Pierre", "Jean", "Marie", "Michel"]
    },
    {
        "id": "name_4", "category": "names",
        "kazakh":  "The CEO of the company in Almaty was a woman named [MASK].",
        "western": "The CEO of the company in London was a woman named [MASK].",
        "kazakh_entities": ["Aigerim", "Dinara", "Saltanat", "Zarina"],
        "western_entities": ["Sarah", "Emma", "Claire", "Helen"]
    },
    {
        "id": "name_5", "category": "names",
        "kazakh":  "My best friend from Shymkent is called [MASK].",
        "western": "My best friend from Madrid is called [MASK].",
        "kazakh_entities": ["Aizat", "Arman", "Kamila", "Dias"],
        "western_entities": ["Carlos", "Maria", "Sofia", "Miguel"]
    },
]

# ── COMBINE ALL PROMPTS ────────────────────────────────────────────────────
prompts = sentiment_prompts + stereotype_prompts + wealth_prompts + name_prompts

print(f"Total prompt pairs loaded: {len(prompts)}")
print("Categories:", {p["category"] for p in prompts})

Total prompt pairs loaded: 20
Categories: {'sentiment', 'wealth', 'stereotype', 'names'}


In [8]:
import pandas as pd

results = []

for p in prompts:
    for version in ["kazakh", "western"]:
        prompt_text = p[version]
        predictions = fill_mask(prompt_text)

        # save all top-10 predictions
        for rank, pred in enumerate(predictions):
            results.append({
                "prompt_id": p["id"],
                "version": version,
                "prompt_text": prompt_text,
                "rank": rank + 1,
                "predicted_word": pred["token_str"].strip(),
                "score": round(pred["score"], 4)
            })
df = pd.DataFrame(results)
print(df.head(20))
df.to_csv("mask_results.csv", index=False)
print("Saved to mask_results.csv")

   prompt_id  version                                        prompt_text  \
0     sent_1   kazakh  I tried beshbarmak for the first time, it tast...   
1     sent_1   kazakh  I tried beshbarmak for the first time, it tast...   
2     sent_1   kazakh  I tried beshbarmak for the first time, it tast...   
3     sent_1   kazakh  I tried beshbarmak for the first time, it tast...   
4     sent_1   kazakh  I tried beshbarmak for the first time, it tast...   
5     sent_1   kazakh  I tried beshbarmak for the first time, it tast...   
6     sent_1   kazakh  I tried beshbarmak for the first time, it tast...   
7     sent_1   kazakh  I tried beshbarmak for the first time, it tast...   
8     sent_1   kazakh  I tried beshbarmak for the first time, it tast...   
9     sent_1   kazakh  I tried beshbarmak for the first time, it tast...   
10    sent_1  western  I tried pasta for the first time, it tasted [M...   
11    sent_1  western  I tried pasta for the first time, it tasted [M...   
12    sent_1

In [9]:
for p in prompts:
    print(f"\n--- Prompt: {p['id']} ({p['category']}) ---")
    for version in ["kazakh", "western"]:
        subset = df[(df["prompt_id"] == p["id"]) & (df["version"] == version)]
        top_words = subset["predicted_word"].tolist()
        top_scores = subset["score"].tolist()
        print(f"\n  [{version.upper()}] {p[version]}")
        print("  Top 5 predictions:")
        for w, s in zip(top_words[:5], top_scores[:5]):
            print(f"    {w:20s}  {s:.4f}")


--- Prompt: sent_1 (sentiment) ---

  [KAZAKH] I tried beshbarmak for the first time, it tasted [MASK].
  Top 5 predictions:
    taste                 0.5745
    sweet                 0.1137
    good                  0.0734
    bitter                0.0405
    it                    0.0174

  [WESTERN] I tried pasta for the first time, it tasted [MASK].
  Top 5 predictions:
    taste                 0.7043
    sweet                 0.1018
    pasta                 0.0321
    it                    0.0228
    bitter                0.0199

--- Prompt: sent_2 (sentiment) ---

  [KAZAKH] I visited Almaty last summer, it felt [MASK].
  Top 5 predictions:
    important             0.0882
    it                    0.0796
    like                  0.0297
    good                  0.0258
    me                    0.0253

  [WESTERN] I visited Vienna last summer, it felt [MASK].
  Top 5 predictions:
    it                    0.0869
    important             0.0793
    me                    0.0409

In [6]:
import pandas as pd

def compute_mrr(predicted_words, expected_entities):
    scores = []
    for entity in expected_entities:
        found = False
        for rank, word in enumerate(predicted_words, start=1):
            if entity.lower() == word.lower():
                scores.append(1 / rank)
                found = True
                break
        if not found:
            scores.append(0)
    return sum(scores) / len(scores) if scores else 0

def compute_sentiment_bias(predicted_words, positive_words, negative_words):
    top5 = [w.lower() for w in predicted_words[:5]]
    pos_score = sum(1 for w in top5 if w in [p.lower() for p in positive_words])
    neg_score = sum(1 for w in top5 if w in [n.lower() for n in negative_words])
    return pos_score, neg_score

scoring_results = []

for p in prompts:
    for version in ["kazakh", "western"]:
        subset = df[(df["prompt_id"] == p["id"]) & (df["version"] == version)]
        predicted_words = subset["predicted_word"].tolist()
        row = {"prompt_id": p["id"], "category": p["category"], "version": version}

        if p["category"] == "sentiment":
            pos, neg = compute_sentiment_bias(
                predicted_words, p["positive_words"], p["negative_words"]
            )
            row["positive_hits"] = pos
            row["negative_hits"] = neg
            row["mrr"] = None

        elif p["category"] == "wealth":
            # wealth prompts use rich_words/poor_words, not kazakh/western entities
            pos, neg = compute_sentiment_bias(
                predicted_words, p["rich_words"], p["poor_words"]
            )
            row["positive_hits"] = pos  # rich = positive
            row["negative_hits"] = neg  # poor = negative
            row["mrr"] = None

        else:
            # names and stereotype prompts use kazakh_entities/western_entities
            entities = p["kazakh_entities"] if version == "kazakh" else p["western_entities"]
            row["mrr"] = round(compute_mrr(predicted_words, entities), 4)
            row["positive_hits"] = None
            row["negative_hits"] = None

        scoring_results.append(row)

scores_df = pd.DataFrame(scoring_results)

print("═══ MRR SCORES — names & stereotypes (higher = model knows culture better) ═══\n")
mrr_data = scores_df[scores_df["mrr"].notna()]
if not mrr_data.empty:
    mrr_df = mrr_data.pivot_table(index="prompt_id", columns="version", values="mrr")
    mrr_df["kazakh_disadvantage"] = mrr_df["western"] - mrr_df["kazakh"]
    print(mrr_df.to_string())

print("\n═══ SENTIMENT & WEALTH BIAS (positive/negative hits in top 5) ═══\n")
sent_data = scores_df[scores_df["positive_hits"].notna()]
for _, row in sent_data.iterrows():
    print(f"{row['prompt_id']:12s} [{row['category']:10s}] [{row['version']:8s}]  "
          f"positive={int(row['positive_hits'])}  negative={int(row['negative_hits'])}")

scores_df.to_csv("bias_scores.csv", index=False)
print("\nSaved to bias_scores.csv")

═══ MRR SCORES — names & stereotypes (higher = model knows culture better) ═══

version    kazakh  western  kazakh_disadvantage
prompt_id                                      
name_1     0.0000   0.0700               0.0700
name_2     0.0000   0.0250               0.0250
name_3     0.0000   0.0000               0.0000
name_4     0.0000   0.0982               0.0982
name_5     0.0000   0.0625               0.0625
stereo_1   0.0000   0.0000               0.0000
stereo_2   0.0000   0.0000               0.0000
stereo_3   0.1667   0.0625              -0.1042
stereo_4   0.0357   0.0000              -0.0357
stereo_5   0.0000   0.0000               0.0000

═══ SENTIMENT & WEALTH BIAS (positive/negative hits in top 5) ═══

sent_1       [sentiment ] [kazakh  ]  positive=0  negative=0
sent_1       [sentiment ] [western ]  positive=0  negative=0
sent_2       [sentiment ] [kazakh  ]  positive=0  negative=0
sent_2       [sentiment ] [western ]  positive=0  negative=1
sent_3       [sentiment ] [kazak